In [120]:
print("Radha")

Radha


In [121]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv

In [122]:
load_dotenv()
model = ChatOpenAI()

In [123]:
# create states
class BatsmanState(TypedDict):
    runs : int 
    balls : int
    fours : int
    sixes : int

    sr : float
    bpb : float
    boundary_percent : float

    summary : str

In [124]:
def calculate_sr(state: BatsmanState):
    sr = state["runs"] / state["balls"] * 100
    return {"sr": sr}

In [125]:
def calculate_bpb(state: BatsmanState):
    bpb = state["balls"] / (state["fours"] + state["sixes"])
    return {"bpb": bpb}

In [126]:
def boundary_percent(state: BatsmanState):
    bp = (
        (state["fours"] * 4 + state["sixes"] * 6)
        / state["runs"]
    ) * 100

    return {"boundary_percent": bp}


In [127]:
def summary(state: BatsmanState):
    prompt = f"""
You are a cricket analyst.

Present these statistics as a beautiful Markdown report.

Statistics:
- Runs: {state['runs']}
- Balls: {state['balls']}
- Strike Rate: {state['sr']:.2f}
- Balls per Boundary: {state['bpb']:.2f}
- Boundary Percentage: {state['boundary_percent']:.2f}%

Use:
- A title
- A markdown table
- A short analysis (2-3 sentences)
"""

    response = model.invoke(prompt).content

    return {"summary": response}

In [128]:
# create node
graph = StateGraph(BatsmanState)

graph.add_node('calculate_sr', calculate_sr)
graph.add_node('calculate_bpb', calculate_bpb)
graph.add_node('calculate_boundary_percent', boundary_percent)
graph.add_node('summary', summary)


# create edges 
graph.add_edge(START, 'calculate_sr')
graph.add_edge(START, 'calculate_bpb')
graph.add_edge(START, 'calculate_boundary_percent')

graph.add_edge('calculate_sr', 'summary')
graph.add_edge('calculate_bpb', 'summary')
graph.add_edge('calculate_boundary_percent', 'summary')

graph.add_edge('summary', END)

workflow = graph.compile()

In [129]:
# excute workflow
intial_state = {
    'runs' : 264,
    'balls' : 173,
    'fours' : 33,
    'sixes' : 9
}

final_state = workflow.invoke(intial_state)
print(final_state['summary'])

# Player's Performance Report

| Statistic           | Value    |
|-------------------|----------|
| Runs               | 264      |
| Balls              | 173      |
| Strike Rate        | 152.60   |
| Balls per Boundary | 4.12     |
| Boundary Percentage| 70.45%   |

The player has shown an outstanding performance with a high strike rate of 152.60 and a boundary percentage of 70.45%. They have been effective in scoring runs at a quick pace, hitting boundaries consistently every 4.12 balls.
